# Interactive map visualization of CML vector data

> This module provides interactive map visualization for Commercial Microwave Link (CML) networks using Folium and GeoPandas. It allows you to explore link geometries on a map with various basemaps and controls.

In [ ]:
#| default_exp maps._vector

In [ ]:
#| export
from typing import Callable

import xarray as xr
import geopandas as gpd
import folium
import folium.plugins
from shapely.geometry import LineString

from raincell import open_cml_sample

In [ ]:
cml = open_cml_sample()

## Default map

Create a Folium map with multiple basemap options (OpenStreetMap, OpenTopoMap, Esri World Imagery), layer controls, fullscreen toggle, geocoder search, and measurement tools that will be used as a default for exploration.

In [ ]:
#| export
def setup_default_map(m: folium.Map = None, show: str = "Esri.WorldImagery") -> folium.Map:
    m = m or folium.Map(tiles=None)
    for tile in ["OpenStreetMap", "OpenTopoMap", "Esri.WorldImagery"]:
        folium.TileLayer(tile, name=tile, show=(tile==show)).add_to(m)
    folium.LayerControl().add_to(m)
    folium.plugins.Fullscreen(position="topright",force_separate_button=True).add_to(m)
    folium.plugins.Geocoder(collapsed=True, add_marker=False).add_to(m)
    folium.plugins.MeasureControl(position="bottomleft").add_to(m)
    return m

In [ ]:
setup_default_map()

## Visualize CML links on an interactive map. 

Each link is drawn as a line between its two antenna sites. Includes a search control to find links by ID. It also includes right-click to copy `cml_id` to clipboard as a python dictionary `({"cml_id": ...})` so the link of interest can easily be selected from the dataset of interest:
```python
ds.sel(**{"cml_id": ...})
```

In [ ]:
#| exporti
def rclick_cp_cml_id(m: folium.Map):
    """ Copy cml_id to clipboard when right clicking on a link """
    js_code = """
    <script>
    document.addEventListener('DOMContentLoaded', function() {
        setTimeout(function() {
            var map = Object.values(window).find(v => v instanceof L.Map);
            map.eachLayer(function(layer) {
                if (layer.feature && layer.feature.properties && layer.feature.properties.cml_id) {
                    layer.on('contextmenu', function(e) {
                        L.DomEvent.stopPropagation(e);
                        L.DomEvent.preventDefault(e);
                        var cmlId = e.target.feature.properties.cml_id;
                        var jsonObj = JSON.stringify({cml_id: cmlId});
                        navigator.clipboard.writeText(jsonObj);
                        console.log('Copied:', jsonObj);
                    });
                }
            });
        }, 1000);
    });
    </script>
    """

    m.get_root().html.add_child(folium.Element(js_code))

In [ ]:
#| export
def explore_links(
    cml: xr.Dataset | xr.DataArray, # CML in OpenSense standard
    default: Callable = setup_default_map, # Function to set up a default map including for example basemaps or controls
    copy_cml_id_on_rclick: bool = True, # Enable copying cml_id to clipboard when right clicking on a link
    **explore_kwargs # kwargs to be passed to the GeoPandas explore [method](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.explore.html) 
    ) -> folium.Map:
    """ Geopandas explore wrapper to visualize CML link structure """
    links_df = cml.cml_id.to_dataframe().drop_duplicates()
    geom = [LineString([(l.site_0_lon, l.site_0_lat), (l.site_1_lon, l.site_1_lat)]) for _, l in links_df.iterrows()]
    links_gdf = gpd.GeoDataFrame(links_df["length"], geometry=geom, crs="EPSG:4326")
    m = links_gdf.explore(tiles=None, **explore_kwargs)
    link_layer, = [c for c in m._children.values() if isinstance(c, folium.features.GeoJson)]
    link_layer.layer_name = "Links"
    if default:
        m = default(m)
    folium.plugins.Search(layer=link_layer, geom_type="LineString", placeholder="CML ID", collapsed=True, search_label="cml_id").add_to(m)
    if copy_cml_id_on_rclick:
        rclick_cp_cml_id(m)
    return m

In [ ]:
explore_links(cml)

## Visualize CML sublinks

Sublinks are colored by frequency using the electromagnetic spectrum coloring (red for lowest frequencies to violet for highest). Sublinks within the same CML are slightly offset for visibility. Optionally shows transmission direction arrows It also supports right-click to copy IDs that can then be used to select the sublink of interest from the dataset (see the example at link visualization section).

In [ ]:
#| exporti
def add_sense(m: folium.Map, layer_name: str = "Sublinks"):
    """ Add directional arrows to sublinks showing the transmission sense (direction from transmitter to receiver) """
    layer = [l for l in m._children.values() if isinstance(l, folium.features.GeoJson) and l.layer_name == layer_name]
    layer = layer[0] if len(layer) == 1 else ValueError(f"{len(layer)} matching layers found. There should be only one matching layer")

    fg = folium.FeatureGroup(name=layer_name + "_sense", show=False).add_to(m)
    for feat in layer.data["features"]:
        coords = feat["geometry"]["coordinates"]
        coords = [[coords[0][1], coords[0][0]], [coords[1][1], coords[1][0]]]
        hidden_pl = folium.PolyLine(coords, weight=0).add_to(fg) # Hiden lines required to add sens over them

        ws = "    "
        arrow = 2*ws + ">" + 2*ws if not feat["properties"]["transmitter"] else 3*ws + "<" + ws
        sense = folium.plugins.PolyLineTextPath(hidden_pl, arrow, repeat=True, offset=8, attributes={"font-weight": "bold", "font-size": "24", "fill": "red"})
        sense.add_to(fg)
    return m

In [ ]:
#| exporti
def rclick_cp_ids(m: folium.Map):
    """ Copy cml_id and sublink_id as a python dict to clipboard when right clicking on a sublink """
    js_code = """
    <script>
    document.addEventListener('DOMContentLoaded', function() {
        setTimeout(function() {
            var map = Object.values(window).find(v => v instanceof L.Map);
            map.eachLayer(function(layer) {
                if (layer.feature && layer.feature.properties && layer.feature.properties.cml_id) {
                    layer.on('contextmenu', function(e) {
                        L.DomEvent.stopPropagation(e);
                        L.DomEvent.preventDefault(e);
                        var cmlId = e.target.feature.properties.cml_id;
                        var sublinkId = e.target.feature.properties.sublink_id;
                        var jsonObj = JSON.stringify({cml_id: cmlId, sublink_id: sublinkId});
                        navigator.clipboard.writeText(jsonObj);
                        console.log('Copied:', jsonObj);
                    });
                }
            });
        }, 1000);
    });
    </script>
    """

    m.get_root().html.add_child(folium.Element(js_code))

In [ ]:
#| export
def explore_sublinks(
    cml: xr.Dataset | xr.DataArray, # CML in OpenSense standard
    default: Callable = setup_default_map, # Function to set up a default map including for example basemaps or controls
    add_transmission_sense: bool = True, # If true it will overlay an arrow over each sublink showing the sense of the signal
    cp_id_on_rclick: bool = True, # Enable copying cml_id and sublink_id when right clicking on a sublink
    **explore_kwargs # kwargs to be passed to the GeoPandas explore [method](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.explore.html) 
    ) -> folium.Map:
    """ Geopandas explore wrapper to visualize CML sublink structure by coloring on frequency """
    sublinks_df = cml[["cml_id", "sublink_id"]].to_dataframe().reset_index()
    sublinks_df = sublinks_df.dropna(subset="frequency").reset_index(drop=True)

    offset = 1e-4 # Offset by ~10m in the equator
    cc = sublinks_df.groupby('cml_id').cumcount() * offset
    for coord in ['site_0_lat', 'site_1_lat', 'site_0_lon', 'site_1_lon']:
        sublinks_df[coord] += cc

    cols = [c for c in sublinks_df if "lat" not in c and "lon" not in c]
    geom = [LineString([(l.site_0_lon, l.site_0_lat), (l.site_1_lon, l.site_1_lat)]) for _, l in sublinks_df.iterrows()]
    sl_gdf = gpd.GeoDataFrame(sublinks_df[cols], geometry=geom, crs="EPSG:4326",)
    if "transmitter" in sl_gdf:
        sl_gdf["transmitter"] = sl_gdf["transmitter"].astype(int)
    explore_kwargs = {"tiles": None, "cmap": "gist_rainbow", "vmin": 8000, "vmax": 18000, "tiles": None, **explore_kwargs}
    m = sl_gdf.explore("frequency", **explore_kwargs)
    sublink_layer, = [layer for name, layer in m._children.items() if name.startswith("geo_json_") and isinstance(layer, folium.features.GeoJson)]
    sublink_layer.layer_name = "Sublinks"

    if add_transmission_sense:
        assert "transmitter" in sl_gdf, ValueError("transmitter coordinate is required to add sense")
        add_sense(m, "Sublinks")
    if default:
        m = default(m)
    if rclick_cp_ids:
        rclick_cp_ids(m)

    return m

In [ ]:
explore_sublinks(cml)